# Exploration: Subsets, Logic Models, and Generalization

In this notebook, we explore the potential of using topic-specific subsets (`animal`, `feel`, `move`, `thing`) to build a "Super Domain" Logic Model.

Our goal is to:
1.  **Construct a Knowledge Base**: Combine vocabulary from multiple subsets to filter the raw `ALL-predicates` dataset.
2.  **Build a LogicModel**: Train a sparse LogicModel on this filtered domain.
3.  **Test Generalization**: Use the massive `gigaWord` test set to evaluate "Coverage" (how many entities do we know?) and "Agreement" (does our model believe the new facts?).

In [1]:
import pickle
import LogicModel as m
import numpy as np
import collections

def load_pickle(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        print(f"File not found: {path}")
        return []

## 1. Load and Analyze Vocabularies

We load the four specific subsets and combine them into a single `super_vocab`. We also load the `ALL` predicates to see what we are working with.

In [2]:
# Load Subsets
animal_subset = set(load_pickle('Predicates/animalSubset.pickle'))
feel_subset = set(load_pickle('Predicates/feelSubset.pickle'))
move_subset = set(load_pickle('Predicates/moveSubset.pickle'))
thing_subset = set(load_pickle('Predicates/thingSubset.pickle'))

print(f"Animal Terms: {len(animal_subset)}")
print(f"Feel Terms:   {len(feel_subset)}")
print(f"Move Terms:   {len(move_subset)}")
print(f"Thing Terms:  {len(thing_subset)}")

# Create Super Vocabulary (Union)
super_vocab = animal_subset | feel_subset | move_subset | thing_subset
print(f"\nTotal Unique Terms in Super Vocabulary: {len(super_vocab)}")

Animal Terms: 5211
Feel Terms:   2637
Move Terms:   2637
Thing Terms:  5401

Total Unique Terms in Super Vocabulary: 7420


In [3]:
# Load Training Data (ALL-predicates)
all_preds = load_pickle('Predicates/ALL-predicates-31994.pickle')
print(f"Total Facts in ALL-predicates: {len(all_preds)}")

# Analyze Vocabulary Overlap
all_vocab = set()
for s, p, o in all_preds:
    if s is not None: all_vocab.add(s)
    if o is not None: all_vocab.add(o)

overlap = super_vocab.intersection(all_vocab)
print(f"Terms in ALL-predicates: {len(all_vocab)}")
print(f"Overlap (Super Vocab found in ALL-predicates): {len(overlap)}")
print(f"Percentage of ALL-predicates vocab covered by Super Vocab: {len(overlap)/len(all_vocab)*100:.2f}%")

Total Facts in ALL-predicates: 31994


Terms in ALL-predicates: 9671
Overlap (Super Vocab found in ALL-predicates): 1374
Percentage of ALL-predicates vocab covered by Super Vocab: 14.21%


## 2. Construct the Training Domain

We filter `ALL-predicates` to keep only the facts where *both* the Subject and Object (if it exists) are present in our `super_vocab`.

This ensures our LogicModel is trained *only* on the concepts we care about (Animals, Feelings, Moves, Things).

In [4]:
filtered_facts = []
filtered_domain_set = set()

for s, p, o in all_preds:
    # Check Subject
    if s not in super_vocab:
        continue
        
    # Check Object (if binary)
    if o is not None and o not in super_vocab:
        continue
        
    # If we pass, add to our filtered list
    filtered_facts.append((s, p, o))
    filtered_domain_set.add(s)
    if o is not None:
        filtered_domain_set.add(o)

print(f"Original Fact Count: {len(all_preds)}")
print(f"Filtered Fact Count: {len(filtered_facts)}")
print(f"Filtered Domain Size: {len(filtered_domain_set)}")

Original Fact Count: 31994
Filtered Fact Count: 3421
Filtered Domain Size: 991


## 3. Build Logic Model (Sparse)

We initialize the sparse `LogicModel` with our filtered facts.

In [5]:
# Prepare Dictionaries for LogicModel
unary_preds_dict = {}
binary_preds_dict = {}

for s, p, o in filtered_facts:
    if o is None:
        if p not in unary_preds_dict: unary_preds_dict[p] = []
        unary_preds_dict[p].append(s)
    else:
        if p not in binary_preds_dict: binary_preds_dict[p] = []
        binary_preds_dict[p].append((s, o))

domain_list = sorted(list(filtered_domain_set))

# Initialize Model
model = m.LogicModel(
    listOfElements=domain_list,
    dictionaryOfUnaryPredicates=unary_preds_dict,
    dictionaryOfBinaryPredicates=binary_preds_dict,
    use_sparse=True
)

print("Building sparse model...")
model.buildAll()
print("Model built successfully!")

Building sparse model...


Model built successfully!


## 4. Run Stability & Coverage Tests

We now introduce the `gigaWord` test set. We want to know:
1.  **Coverage**: How much of the "real world" (GigaWord) does our limited domain model understand?
2.  **Agreement**: For facts involving known entities, does our model consider them possible (Probability > 0)?

In [6]:
# Load Test Set
giga_preds = load_pickle('Predicates/gigaWord-testSet-154648.pickle')
print(f"Total Test Facts (GigaWord): {len(giga_preds)}")

# Metrics
known_entities_count = 0
known_facts_count = 0
agreement_count = 0
total_test_facts = len(giga_preds)

matches = []
misses = []

print("Running evaluation... (this might take a moment)")

for s, p, o in giga_preds:
    # Check if we know the entities
    s_known = s in filtered_domain_set
    o_known = (o is None) or (o in filtered_domain_set)
    
    if s_known and o_known:
        known_facts_count += 1
        
        # Query the model
        # Since we just want to know if it's *true* in the model (exists in the training set),
        # we can check the dictionaries directly for O(1) speed, OR check the tensor.
        # But since we built a probabilistic model (or at least a model structure), 
        # let's simulate a 'query'. 
        # Note: Our current model build assumes provided facts are P=1.0.
        # Facts NOT provided are P=0.0 (Closed World Assumption for sparse).
        
        # Faster check using the raw dictionaries we built the model with:
        is_true = False
        if o is None:
            if p in unary_preds_dict and s in unary_preds_dict[p]:
                is_true = True
        else:
            if p in binary_preds_dict and (s, o) in binary_preds_dict[p]:
                is_true = True
        
        if is_true:
            agreement_count += 1
            if len(matches) < 5:
                matches.append((s, p, o))
    else:
        if len(misses) < 5:
            misses.append((s, p, o))

print("Evaluation Complete.")

Total Test Facts (GigaWord): 154648
Running evaluation... (this might take a moment)
Evaluation Complete.


## 5. Results & Analysis

In [7]:
coverage_pct = (known_facts_count / total_test_facts) * 100
agreement_pct = (agreement_count / known_facts_count) * 100 if known_facts_count > 0 else 0

print(f"=== Results ===")
print(f"Test Set Size:      {total_test_facts}")
print(f"Facts Covered:      {known_facts_count} (Entities known to model)")
print(f"Coverage:           {coverage_pct:.2f}%")
print(f"Facts Agreed Upon:  {agreement_count} (Model says TRUE)")
print(f"Agreement Rate:     {agreement_pct:.2f}% (of covered facts)")

print(f"\n--- Examples of Agreed Facts (Model knew these) ---")
for m in matches:
    print(m)

print(f"\n--- Examples of Missed Facts (Model didn't know entities) ---")
for m in misses:
    print(m)

=== Results ===
Test Set Size:      154648
Facts Covered:      10311 (Entities known to model)
Coverage:           6.67%
Facts Agreed Upon:  187 (Model says TRUE)
Agreement Rate:     1.81% (of covered facts)

--- Examples of Agreed Facts (Model knew these) ---
('people', 'is_think', None)
('official', 'is_said', None)
('official', 'is_said', None)
('official', 'is_said', None)
('official', 'is_said', None)

--- Examples of Missed Facts (Model didn't know entities) ---
('tributes', 'poured', 'statement')
('smith', 'died', 'statement')
('department', 'issued', 'statement')
('smith', 'left', 'impression')
('mccurry', 'said', 'condolences')
